# Data Overview
## OHLCV Price Data Analysis

### Objectives:
- Load and examine ETH/USD OHLCV data
- Understand data structure and types
- Check for missing values and data quality
- Initial statistical summary
- Data range and frequency analysis

### Data Sources:
- CSV file: `../../../data/Eth_OHLCV.csv`
- JSON file: `../../../data/Eth_OHLCV.json`
- SQLite DB: `../../../data/ETH.db`

📁 Current notebook: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\notebooks\01_eda_exploratory_data_analysis
📁 OHLCV root: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv
📁 Utils path: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\utils
✅ Added utils path: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\utils

📁 Utils folder contents:
   - data_loader.py
   - trading_helpers.py
   - visualizations.py
   - __init__.py

✅ All utilities imported successfully!

✅ SETUP COMPLETE
⚠️  CSV file not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\Eth_OHLCV.csv
⚠️  Database not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\ETH.db
⚠️  JSON file not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereu

""


In [ ]:
# Initialize data loader
loader = DataLoader()

# Load data from CSV
df = loader.load_from_csv()

if df.empty:
    print("CSV not found, trying JSON...")
    df = loader.load_from_json()

if df.empty:
    print("JSON not found, trying database...")
    df = loader.load_from_db()

print(f"✅ Data loaded successfully!")
print(f"📊 Shape: {df.shape}")
print(f"📅 Date range: {df.index.min()} to {df.index.max()}")
print(f"📈 Total periods: {len(df)}")
df.head(10)

In [ ]:
# Data info
print("=" * 60)
print("📋 DATA INFORMATION")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("📊 DATA TYPES")
print("=" * 60)
print(df.dtypes)

print("\n" + "=" * 60)
print("🔍 NULL VALUES")
print("=" * 60)
print(df.isnull().sum())

In [ ]:
# Statistical summary
print("=" * 60)
print("📈 STATISTICAL SUMMARY")
print("=" * 60)
df.describe()

In [ ]:
# Check for duplicates
duplicates = df.index.duplicated().sum()
print(f"🔄 Duplicate timestamps: {duplicates}")

if duplicates > 0:
    print("\n⚠️ Duplicate timestamps found:")
    print(df[df.index.duplicated(keep=False)].sort_index().head())
    df = df[~df.index.duplicated(keep='first')]
    print(f"\n✅ Removed duplicates. New shape: {df.shape}")

In [ ]:
# Data quality check
print("=" * 60)
print("🔍 DATA QUALITY CHECK")
print("=" * 60)

# Check for zero or negative values
for col in ['open', 'high', 'low', 'close', 'volume']:
    if col in df.columns:
        invalid = (df[col] <= 0).sum()
        print(f"{col}: {invalid} invalid values (<=0)")

# Check OHLC logic
invalid_ohlc = ((df['high'] < df['low']) | 
               (df['high'] < df['open']) | 
               (df['high'] < df['close']) |
               (df['low'] > df['open']) | 
               (df['low'] > df['close'])).sum()
print(f"\n⚠️ Invalid OHLC relationships: {invalid_ohlc}")

if invalid_ohlc > 0:
    print("\nInvalid rows:")
    invalid_mask = ((df['high'] < df['low']) | 
                   (df['high'] < df['open']) | 
                   (df['high'] < df['close']) |
                   (df['low'] > df['open']) | 
                   (df['low'] > df['close']))
    print(df[invalid_mask].head())